In [1]:
# importamos librerías
# para manipulación de datos
import pandas as pd
import numpy as np
# para trabajar con APIs
import requests
# para gestionar archivos y directorios
import os
import zipfile
# para manipular pdfs
import camelot
# para trabajar con fechas
from datetime import datetime

In [ ]:
url_cmadrid_anual = "https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_datos_historico"
url_cmadrid_mes_curso = "https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_datos_mes"
url_cmadrid_dia_curso = "https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_datos_dia"
url_cmadrid_estaciones = "https://datos.comunidad.madrid/api/3/action/package_show?id=calidad_aire_estaciones"
url_madrid = "https://datos.madrid.es/egob/catalogo/keyword/aire.json"

In [ ]:
# extraemos datos históricos de la API de la Comunidad de Madrid
# petición a la API
response_cmadrid_anual = requests.get(url_cmadrid_anual)
# verificamos si la respuesta es exitosa
if response_cmadrid_anual.status_code == 200:
    # convertimos la respuesta a JSON
    data_cmadrid_anual = response_cmadrid_anual.json()
    # obtenemos la key donde se encuentran los datos que necesitamos
    todos_annos = data_cmadrid_anual.get("result", {}).get("resources")
    # lista vacía para almacenar los nombres de los archivos
    archivos = []
    # recorremos la lista de diccionarios y extraemos los nombres de los archivos
    for anno in todos_annos:
        archivos.append(anno["name"])
    archivos_guardados = 0 #contador de archivos guardados
    archivos_totales = len(archivos) # total de archivos a descargar
    # iteramos sobre la lista de archivos y descargamos cada dataset
    for archivo in archivos:
        for anno in todos_annos:
            if anno["name"] == archivo:
                datos_anno = anno.get("url")
                formato = anno.get("format")
                response_anno = requests.get(datos_anno)
                # Verificar si la descarga fue exitosa
                if response_anno.status_code == 200:
                # Guardar el archivo en tu computadora
                    with open(f'../data/cmadrid_{archivo}.{formato}', 'wb') as file:
                        file.write(response_anno.content)
                        archivos_guardados +=1
                else:
                    print(f"No se pudo descargar el archivo {archivo}. Código de estado: {response_anno.status_code}")
    print(f"{archivos_guardados} de {archivos_totales} archivos guardados exitosamente.")

22 de 22 archivos guardados exitosamente.


In [7]:
# extraemos datos del mes en curso de la API de la Comunidad de Madrid
# petición a la API
response_cmadrid_mes_curso = requests.get(url_cmadrid_mes_curso)
# verificamos si la respuesta es exitosa
if response_cmadrid_mes_curso.status_code == 200:
    # convertimos la respuesta a JSON
    data_cmadrid_mes_curso = response_cmadrid_mes_curso.json()
    # obtenemos la key donde se encuentran los datos que necesitamos
    mes_curso = data_cmadrid_mes_curso.get("result", {}).get("resources")
    # iteramos sobre la lista de diccionarios para extraer solo los archivos csv
    for elemento in mes_curso:
        if elemento["format"] == "CSV":
            datos_mes = elemento.get("url")
            response = requests.get(datos_mes)
            hoy =  datetime.now().date()  
            # Verificar si la descarga fue exitosa
            if response.status_code == 200:
                # Guardar el archivo en tu computadora
                with open(f'../data/cmadrid_{hoy.month}_{hoy.year}.csv', 'wb') as file:
                    file.write(response.content)
                    print("Archivo guardado exitosamente.")
            else:
                    print(f"No se pudo descargar el archivo. Código de estado: {response.status_code}")

Archivo guardado exitosamente.


In [9]:
# extraemos datos del día en curso de la API de la Comunidad de Madrid
# petición a la API
response_cmadrid_dia_curso = requests.get(url_cmadrid_dia_curso)
# verificamos si la respuesta es exitosa
if response_cmadrid_dia_curso.status_code == 200:
    # convertimos la respuesta a JSON
    data_cmadrid_dia_curso = response_cmadrid_dia_curso.json()
    # obtenemos la key donde se encuentran los datos que necesitamos
    dia_curso = data_cmadrid_dia_curso.get("result", {}).get("resources")
    # iteramos sobre la lista de diccionarios para extraer solo los archivos csv
    for elemento in dia_curso:
        if elemento["format"] == "CSV":
            datos_dia = elemento.get("url")
            response = requests.get(datos_dia)
            hoy =  datetime.now().date()  
            # Verificar si la descarga fue exitosa
            if response.status_code == 200:
                # Guardar el archivo en tu computadora
                with open(f'../data/cmadrid_{hoy}.csv', 'wb') as file:
                    file.write(response.content)
                    print("Archivo guardado exitosamente.")
            else:
                    print(f"No se pudo descargar el archivo. Código de estado: {response.status_code}")

Archivo guardado exitosamente.


In [12]:
# extraemos datos de estaciones de la API de la Comunidad de Madrid
# petición a la API
response_cmadrid_estaciones = requests.get(url_cmadrid_estaciones)
# verificamos si la respuesta es exitosa
if response_cmadrid_estaciones.status_code == 200:
    # convertimos la respuesta a JSON
    data_cmadrid_estaciones = response_cmadrid_estaciones.json()
    # obtenemos la key donde se encuentran los datos que necesitamos
    estaciones = data_cmadrid_estaciones.get("result", {}).get("resources")
    # iteramos sobre la lista de diccionarios para extraer solo los archivos csv
    for elemento in estaciones:
        if elemento["format"] == "CSV":
            datos_estaciones = elemento.get("url")
            response = requests.get(datos_estaciones)
            hoy =  datetime.now().date()  
            # Verificar si la descarga fue exitosa
            if response.status_code == 200:
                # Guardar el archivo en tu computadora
                with open(f'../data/cmadrid_estaciones.csv', 'wb') as file:
                    file.write(response.content)
                    print("Archivo guardado exitosamente.")
            else:
                    print(f"No se pudo descargar el archivo. Código de estado: {response.status_code}")

Archivo guardado exitosamente.


In [4]:
# Ruta del archivo PDF
archivo_pdf = "../data/raw/cmadrid_Descripción datos de contaminantes.PDF"

# Extraer tablas
tablas = camelot.read_pdf(archivo_pdf, pages="3")

# Verificar si se extrajeron tablas
if len(tablas) > 0:
        tablas[0].to_csv(f"../data/raw/datos_contaminantes_cmadrid.csv")  # Guardar como CSV

else:
    print("No se encontraron tablas en la página indicada.")



In [5]:
# Ruta del archivo PDF
archivo_pdf = "../data/raw/Interprete_ficheros_ calidad_ del_ aire_global.pdf"

# Extraer tablas
tablas = camelot.read_pdf(archivo_pdf, pages="7")

# Verificar si se extrajeron tablas
if len(tablas) > 0:
        tablas[0].to_csv(f"../data/raw/datos_contaminantes_madrid.csv")  # Guardar como CSV

else:
    print("No se encontraron tablas en la página indicada.")

In [6]:
datos_contaminantes_cmadrid = pd.read_csv("../data/raw/datos_contaminantes_cmadrid.csv")
datos_contaminantes_cmadrid

,CÓDIGO \nMAGNITUD,DESCRIPCIÓN MAGNITUD,CÓDIGO \nTÉCNICA \nDE MEDIDA,DESCRIPCIÓN TÉCNICA \nDE MEDIDA,UNIDAD,DESCRIPCIÓN UNIDAD
0,1,Dióxido de azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cúbico
1,6,Monóxido de carbono,48,Espectrometría infrarroja no \ndispersiva,mg/m³,miligramos por metro cúbico
2,7,Monóxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
3,8,Dióxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
4,9,"Partículas en suspensión < PM2,5",49,Absorción beta,µg/m³,microgramos por metro cubico
5,10,Partículas en suspensión < PM10,49,Absorción beta,µg/m³,microgramos por metro cubico
6,11,Partículas en suspensión < PM1,49,Absorción beta,µg/m³,microgramos por metro cubico
7,12,Óxidos de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
8,14,Ozono,6,Absorción ultravioleta,µg/m³,microgramos por metro cubico
9,14,Ozono Quimioluminiscencia,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico


In [7]:
datos_contaminantes_madrid = pd.read_csv("../data/raw/datos_contaminantes_madrid.csv")
datos_contaminantes_madrid

,Cod.,Mágnitud,Abrevia.,Unidad,Código \nTécnica,Técnica de medida
0,1,Dióxido de Azufre,SO2,µg/m 3,38,Fluorescencia ultravioleta
1,6,Monóxido de Carbono,CO,mg/m3,48,Absorción infrarroja
2,7,Monóxido de Nitrógeno,NO,µg/m3,8,Quimioluminiscencia
3,8,Dióxido de Nitrógeno,NO2,µg/m 3,8,Quimioluminiscencia
4,9,Partículas < 2.5 µm,PM2.5,µg/m3,47,Microbalanza/Espectrometría*
5,10,Partículas < 10 µm,PM10,µg/m3,47,Microbalanza/Espectrometría*
6,12,Óxidos de Nitrógeno,NOx,µg/m3,8,Quimioluminiscencia
7,14,Ozono,O3,µg/m 3,6,Absorción ultravioleta
8,20,Tolueno,TOL,µg/m3,59,Cromatografía de gases
9,30,Benceno,BEN,µg/m3,59,Cromatografía de gases


In [ ]:
lista_titulos = ["Calidad del aire. Estaciones de control", 
                 "Calidad del aire. Datos en tiempo real acumulado", 
                 "Calidad del aire. Datos horarios desde 2001"
                ]

In [ ]:
# Hacemos la petición y comprobamos el estado
hoy =  datetime.now().date()  
response = requests.get(url_madrid)
if response.status_code == 200:
    data = response.json()
    elementos = data.get("result", {}).get("items")
    for elemento in elementos:
        if elemento["title"] == "Calidad del aire. Estaciones de control":
            datos_est = elemento.get("distribution")
            for i in datos_est:
                print(i.get("title"))
                formato = i.get("format",{}).get("value").split('/')[-1]
                if formato.lower() == "csv":
                    archivo = i.get("accessURL")
                    print(f"Descargando: {archivo} como madrid_{elemento["title"]}.{formato}")
                    response_est = requests.get(archivo)
                    if response_est.status_code == 200:
                        with open(f'../datos/madrid_{elemento["title"]}.{formato}', 'wb') as file:
                            file.write(response_est.content)
                            print("Archivo guardado exitosamente.")
                    else:
                        print(f"No se pudo descargar el archivo {elemento["title"]}. Código de estado: {response.status_code}")
        elif elemento["title"] == "Calidad del aire. Datos en tiempo real acumulado":
            datos_tra = elemento.get("distribution")
            for i in datos_tra:
                formato = i.get("format",{}).get("value").split('/')[-1]
                if "csv" in i.get("title"):
                    archivo = i.get("accessURL")
                    print(f"Descargando: {archivo} como madrid_{elemento["title"]}.{formato}")
                    response_tra = requests.get(archivo)
                    if response_tra.status_code == 200:
                        with open(f'../datos/madrid_{hoy}.{formato}', 'wb') as file:
                            file.write(response_tra.content)
                            print("Archivo guardado exitosamente.")
                    else:
                        print(f"No se pudo descargar el archivo {elemento["title"]}. Código de estado: {response.status_code}")
        elif elemento["title"] == "Calidad del aire. Datos horarios desde 2001":
            archivos_dh = []
            datos = elemento.get("distribution")
            for i in datos:
                formato = i.get("format",{}).get("value").split('/')[-1]
                archivo = i.get("accessURL")
                print(f"Descargando: {archivo} como madrid_{i["title"]}.{formato}")
                response_dh = requests.get(archivo)
                if response_dh.status_code == 200:
                    with open(f'../datos/madrid_{i["title"]}.{formato}', 'wb') as file:
                        file.write(response_dh.content)
                        archivos_dh.append(i["title"])
                        print("Archivo guardado exitosamente.")
                else:
                    print(f"No se pudo descargar el archivo {i["title"]}. Código de estado: {response.status_code}")



In [ ]:
import zipfile
import os

# Diccionarios para almacenar listas de DataFrames y nombres de CSV
listas_dfs = {}
listas_csv = {}

# Crear una carpeta para guardar los archivos concatenados si no existe
carpeta_salida = "../datos"
os.makedirs(carpeta_salida, exist_ok=True)

for archivo in archivos_dh:
    ruta_zip = f"../datos/madrid_{archivo}.zip"  # Construir la ruta al archivo ZIP
    listas_dfs[archivo] = []  # Inicializar una lista de DataFrames para este archivo
    listas_csv[archivo] = []  # Inicializar una lista de nombres de CSV para este archivo

    # Abrir el ZIP y cargar los CSV
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        listas_csv[archivo] = [nombre for nombre in zip_ref.namelist() if nombre.endswith(".csv")]
        
        for nombre_csv in listas_csv[archivo]:
            with zip_ref.open(nombre_csv) as f:
                df = pd.read_csv(f, sep=";", index_col=0)
                listas_dfs[archivo].append(df)

    # Concatenar los DataFrames de la lista en uno solo
    df_concatenado = pd.concat(listas_dfs[archivo], ignore_index=True)

    # Guardar el DataFrame concatenado en un archivo CSV
    ruta_salida = os.path.join(carpeta_salida, f"madrid_{archivo}.csv")
    df_concatenado.to_csv(ruta_salida, sep=";", index=False)

    print(f"Archivo concatenado guardado en: {ruta_salida}")
   
    # Eliminar el archivo ZIP
    os.remove(ruta_zip)
    print(f"Archivo ZIP eliminado: {ruta_zip}")
               